## setup

In [4]:
import os, re, cv2, pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

In [2]:
original_csv_path = "./data/teknofest/data-original.csv"
original_images_folder = "./data/teknofest/images/original"
original_recreated_annotations_folder = "./data/teknofest/annotations/original-recreated"
padded_resized_grayscaled_images_folder = "./data/teknofest/images/padded-resized-grayscaled"
padded_resized_grayscaled_annotations_folder = "./data/teknofest/annotations/padded-resized-grayscaled"
padded_resized_csv_path = "./data/teknofest/data-padded-resized.csv"

In [3]:
df = pd.read_csv(original_csv_path)
df.head()

,filename,a_width,a_x,a_y,b_width,b_x,b_y,c_width,c_x,c_y,line_height,line_x,line_y,KTO
0,1 (1).jpg,408,1209,1575,693,1624,1732,2254,442,1751,1568,1620,178,0.488408
1,1 (2).jpg,619,1739,1722,395,1326,1466,1945,688,1791,1575,1730,213,0.521298
2,1 (3).jpg,150,362,713,341,515,735,932,47,810,905,513,44,0.526502
3,1 (4).jpg,432,1070,1819,782,1509,1994,2701,180,2292,1920,1505,369,0.449316
4,1 (5).jpg,590,804,1611,597,1401,1652,2015,339,1920,2110,1394,190,0.589226


## preprocess teknofest data

### update csv file to adapt changes (padding & resizing)

In [4]:
def get_image_size(image_path):
    """Get the width and height of an image."""
    with Image.open(image_path) as img:
        return img.size  # Returns (width, height)

def transform_coordinates(original_width, original_height, target_size=512):
    """
    Calculate transformation parameters for center padding to square and resize to target_size.
    
    Args:
        original_width: Original image width
        original_height: Original image height
        target_size: Final square size (default 512)
    
    Returns:
        scale_factor: Factor to scale coordinates and dimensions
        offset_x: X offset after center padding
        offset_y: Y offset after center padding
    """
    # Step 1: Center padding to square
    max_dim = max(original_width, original_height)
    
    # Calculate padding offsets (how much to add on each side)
    pad_x = (max_dim - original_width) // 2
    pad_y = (max_dim - original_height) // 2
    
    # Step 2: Resize square to target_size
    scale_factor = target_size / max_dim
    
    return scale_factor, pad_x, pad_y

def update_annotations(csv_path, images_folder, output_csv_path, target_size=512):
    """
    Update annotations CSV with transformed coordinates.
    
    Args:
        csv_path: Path to the original CSV file
        images_folder: Path to folder containing original images
        output_csv_path: Path to save updated CSV
        target_size: Final image size (default 512)
    """
    # Load the CSV
    df = pd.read_csv(csv_path)
    
    # Create a copy for modifications
    updated_df = df.copy()
    
    print(f"Processing {len(df)} annotations...")
    
    for idx, row in df.iterrows():
        filename = row['filename']
        image_path = os.path.join(images_folder, filename)
        
        if not os.path.exists(image_path):
            print(f"Warning: Image {filename} not found in {images_folder}")
            continue
        
        # Get original image dimensions
        original_width, original_height = get_image_size(image_path)
        
        # Calculate transformation parameters
        scale_factor, pad_x, pad_y = transform_coordinates(
            original_width, original_height, target_size
        )
        
        # Update horizontal lines (a, b, c)
        for line_name in ['a', 'b', 'c']:
            # Update x coordinate (add padding offset, then scale)
            x_col = f'{line_name}_x'
            if pd.notna(row[x_col]):
                new_x = (row[x_col] + pad_x) * scale_factor
                updated_df.at[idx, x_col] = new_x
            
            # Update y coordinate (add padding offset, then scale)
            y_col = f'{line_name}_y'
            if pd.notna(row[y_col]):
                new_y = (row[y_col] + pad_y) * scale_factor
                updated_df.at[idx, y_col] = new_y
            
            # Update width (just scale, no offset needed)
            width_col = f'{line_name}_width'
            if pd.notna(row[width_col]):
                new_width = row[width_col] * scale_factor
                updated_df.at[idx, width_col] = new_width
        
        # Update vertical line
        if pd.notna(row['line_x']):
            new_line_x = (row['line_x'] + pad_x) * scale_factor
            updated_df.at[idx, 'line_x'] = new_line_x
        
        if pd.notna(row['line_y']):
            new_line_y = (row['line_y'] + pad_y) * scale_factor
            updated_df.at[idx, 'line_y'] = new_line_y
        
        if pd.notna(row['line_height']):
            new_line_height = row['line_height'] * scale_factor
            updated_df.at[idx, 'line_height'] = new_line_height
        
        if (idx + 1) % 100 == 0:
            print(f"Processed {idx + 1}/{len(df)} images...")
    
    # Save updated CSV
    updated_df.to_csv(output_csv_path, index=False)
    print(f"Updated annotations saved to: {output_csv_path}")
    
    return updated_df

def verify_transformation(original_csv, updated_csv, sample_indices=[0, 1, 2]):
    """
    Verify the transformation by comparing a few samples.
    """
    orig_df = pd.read_csv(original_csv)
    updated_df = pd.read_csv(updated_csv)
    
    print("\n=== Transformation Verification ===")
    for idx in sample_indices:
        if idx < len(orig_df):
            print(f"\nSample {idx + 1} (filename: {orig_df.iloc[idx]['filename']}):")
            print("Original coordinates:")
            print(f"  a: x={orig_df.iloc[idx]['a_x']:.2f}, y={orig_df.iloc[idx]['a_y']:.2f}, width={orig_df.iloc[idx]['a_width']:.2f}")
            print(f"  line: x={orig_df.iloc[idx]['line_x']:.2f}, y={orig_df.iloc[idx]['line_y']:.2f}, height={orig_df.iloc[idx]['line_height']:.2f}")
            
            print("Updated coordinates:")
            print(f"  a: x={updated_df.iloc[idx]['a_x']:.2f}, y={updated_df.iloc[idx]['a_y']:.2f}, width={updated_df.iloc[idx]['a_width']:.2f}")
            print(f"  line: x={updated_df.iloc[idx]['line_x']:.2f}, y={updated_df.iloc[idx]['line_y']:.2f}, height={updated_df.iloc[idx]['line_height']:.2f}")


In [5]:
updated_df = update_annotations(original_csv_path, original_images_folder, padded_resized_csv_path)
verify_transformation(original_csv_path, padded_resized_csv_path)

Processing 102 annotations...
Processed 100/102 images...
Updated annotations saved to: ./data/teknofest/data-padded-resized.csv

=== Transformation Verification ===

Sample 1 (filename: 1 (1).jpg):
Original coordinates:
  a: x=1209.00, y=1575.00, width=408.00
  line: x=1620.00, y=178.00, height=1568.00
Updated coordinates:
  a: x=209.83, y=262.50, width=68.00
  line: x=278.33, y=29.67, height=261.33

Sample 2 (filename: 1 (2).jpg):
Original coordinates:
  a: x=1739.00, y=1722.00, width=619.00
  line: x=1730.00, y=213.00, height=1575.00
Updated coordinates:
  a: x=294.53, y=291.65, width=104.84
  line: x=293.01, y=36.08, height=266.75

Sample 3 (filename: 1 (3).jpg):
Original coordinates:
  a: x=362.00, y=713.00, width=150.00
  line: x=513.00, y=44.00, height=905.00
Updated coordinates:
  a: x=168.96, y=355.18, width=70.01
  line: x=239.43, y=42.94, height=422.39


/tmp/ipykernel_91501/1288364243.py:72: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '209.83333333333331' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  updated_df.at[idx, x_col] = new_x
/tmp/ipykernel_91501/1288364243.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '262.5' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  updated_df.at[idx, y_col] = new_y
/tmp/ipykernel_91501/1288364243.py:78: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '288.66666666666663' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  updated_df.at[idx, y_col] = new_y
/tmp/ipykernel_91501/1288364243.py:84: FutureWarning: Setting an item of incompa

### draw annotations on images

In [9]:
def draw_horizontal_line(image, x, y, width, color, thickness=2, label="", font_scale=0.8, text_thickness=2):
    """
    Draw a horizontal line with centered label and length at the end.
    
    Args:
        image: OpenCV image array
        x, y: Starting coordinates of the line
        width: Width of the line
        color: Color tuple (B, G, R) for OpenCV
        thickness: Line thickness
        label: Label text (e.g., 'a', 'b', 'c')
        font_scale: Adaptive font scale based on image size
        text_thickness: Adaptive text thickness based on image size
    """
    if pd.isna(x) or pd.isna(y) or pd.isna(width):
        return image
    
    # Convert to integers
    x, y, width = int(x), int(y), int(width)
    
    # Draw the horizontal line
    start_point = (x, y)
    end_point = (x + width, y)
    cv2.line(image, start_point, end_point, color, thickness)
    
    font = cv2.FONT_HERSHEY_SIMPLEX
    
    # Add label at the center of the line
    if label:
        center_x = x + width // 2
        center_y = y - int(15 * font_scale)  # Scale the offset based on font size
        
        # Ensure text doesn't go outside image boundaries
        center_y = max(int(25 * font_scale), center_y)
        
        # Get text size for centering
        (text_width, text_height), baseline = cv2.getTextSize(label, font, font_scale, text_thickness)
        text_x = center_x - text_width // 2
        text_x = max(5, min(text_x, image.shape[1] - text_width - 5))
        
        # Draw label
        cv2.putText(image, label, (text_x, center_y), font, font_scale, color, text_thickness)
    
    # Add length measurement at the end of the line
    length_text = f"{int(width)}"
    end_x = x + width + int(10 * font_scale)  # Scale the offset based on font size
    end_y = y + int(5 * font_scale)  # Scale the offset based on font size
    
    # Ensure text doesn't go outside image boundaries
    (length_width, length_height), baseline = cv2.getTextSize(length_text, font, font_scale, text_thickness)
    end_x = min(end_x, image.shape[1] - length_width - 5)
    end_y = max(length_height + 5, min(end_y, image.shape[0] - 5))
    
    # Draw length
    cv2.putText(image, length_text, (end_x, end_y), font, font_scale, color, text_thickness)
    
    return image

def draw_vertical_line(image, x, y, height, color, thickness=2):
    """
    Draw a vertical line without labels or measurements.
    
    Args:
        image: OpenCV image array
        x, y: Starting coordinates of the line
        height: Height of the line
        color: Color tuple (B, G, R) for OpenCV
        thickness: Line thickness
    """
    if pd.isna(x) or pd.isna(y) or pd.isna(height):
        return image
    
    # Convert to integers
    x, y, height = int(x), int(y), int(height)
    
    # Draw the vertical line only
    start_point = (x, y)
    end_point = (x, y + height)
    cv2.line(image, start_point, end_point, color, thickness)
    
    return image

def annotate_single_image(image_path, row, output_path):
    """
    Annotate a single image with all the lines and measurements.
    
    Args:
        image_path: Path to the input image
        row: Pandas row containing annotation data
        output_path: Path to save the annotated image
    """
    # Load image
    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not load image {image_path}")
        return False
    
    # Calculate adaptive scaling based on image size
    height, width = image.shape[:2]
    max_dimension = max(height, width)
    
    # Adaptive line thickness (your existing logic)
    line_thickness = round(max_dimension / 250)
    line_thickness = max(1, line_thickness)  # Ensure minimum thickness of 1
    
    # Adaptive font scaling - scale font size based on image size
    # Base font scale of 0.8 for ~512px images, scale proportionally
    base_font_scale = 0.8
    base_dimension = 512
    font_scale = base_font_scale * (max_dimension / base_dimension)
    font_scale = max(0.4, min(font_scale, 2.0))  # Clamp between 0.4 and 2.0
    
    # Adaptive text thickness - scale text thickness based on image size
    # Base text thickness of 2 for ~512px images, scale proportionally
    base_text_thickness = 2
    text_thickness = round(base_text_thickness * (max_dimension / base_dimension))
    text_thickness = max(1, text_thickness)  # Ensure minimum thickness of 1
    
    # Define colors (BGR format for OpenCV)
    colors = {
        'a': (0, 255, 0),      # Green
        'b': (255, 0, 0),      # Blue  
        'c': (0, 0, 255),      # Red
        'line': (255, 255, 0)  # Cyan
    }
    
    # Draw vertical line FIRST (so it goes to the back)
    x = row['line_x']
    y = row['line_y']
    height = row['line_height']
    
    image = draw_vertical_line(
        image, x, y, height,
        colors['line'],
        thickness=line_thickness
    )
    
    # Draw horizontal lines (a, b, c) ON TOP of vertical line
    for line_name in ['a', 'b', 'c']:
        x = row[f'{line_name}_x']
        y = row[f'{line_name}_y']
        width = row[f'{line_name}_width']
        
        image = draw_horizontal_line(
            image, x, y, width, 
            colors[line_name], 
            thickness=line_thickness,
            label=line_name,
            font_scale=font_scale,
            text_thickness=text_thickness
        )
    
    # Add KTO value if available with adaptive font scaling
    if 'KTO' in row and pd.notna(row['KTO']):
        kto_text = f"KTO: {row['KTO']:.2f}"
        font = cv2.FONT_HERSHEY_SIMPLEX
        # Scale KTO font size (typically larger than line labels)
        kto_font_scale = font_scale * 1.5  # Make KTO text 50% larger than line labels
        kto_font_scale = min(kto_font_scale, 2.5)  # Cap the maximum size
        kto_thickness = max(text_thickness + 1, 3)  # Make KTO text slightly thicker
        color = (255, 255, 255)  # White
        
        # Position at bottom-right corner with adaptive spacing
        (text_width, text_height), baseline = cv2.getTextSize(kto_text, font, kto_font_scale, kto_thickness)
        margin = int(20 * font_scale)  # Scale margin based on font size
        text_x = image.shape[1] - text_width - margin
        text_y = image.shape[0] - margin
        
        # Draw KTO text
        cv2.putText(image, kto_text, (text_x, text_y), font, kto_font_scale, color, kto_thickness)
    
    # Save annotated image
    success = cv2.imwrite(output_path, image)
    if not success:
        print(f"Error: Could not save image to {output_path}")
        return False
    
    return True

def annotate_images_from_csv(csv_path, images_folder, output_folder):
    """
    Main function to annotate all images based on CSV data.
    
    Args:
        csv_path: Path to the updated CSV file with annotations
        images_folder: Path to folder containing processed images
        output_folder: Path to folder where annotated images will be saved
    """
    # Load CSV
    df = pd.read_csv(csv_path)
    print(f"Loaded {len(df)} annotations from CSV")
    
    # Create output folder if it doesn't exist
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    
    successful_annotations = 0
    failed_annotations = 0
    
    print(f"Starting annotation process...")
    print(f"Input folder: {images_folder}")
    print(f"Output folder: {output_folder}")
    print("-" * 50)
    
    for idx, row in df.iterrows():
        filename = row['filename']
        
        # Input and output paths
        input_path = os.path.join(images_folder, filename)
        output_path = os.path.join(output_folder, filename)
        
        # Check if input image exists
        if not os.path.exists(input_path):
            print(f"Warning: Image {filename} not found in {images_folder}")
            failed_annotations += 1
            continue
        
        # Annotate the image
        success = annotate_single_image(input_path, row, output_path)
        
        if success:
            successful_annotations += 1
            if (successful_annotations) % 50 == 0:
                print(f"Progress: {successful_annotations}/{len(df)} images annotated")
        else:
            failed_annotations += 1
            print(f"Failed to annotate: {filename}")
    
    print("-" * 50)
    print(f"Annotation complete!")
    print(f"Successfully annotated: {successful_annotations} images")
    print(f"Failed annotations: {failed_annotations} images")
    print(f"Annotated images saved in: {output_folder}")

#### draw on preprocessed images

In [ ]:
annotate_images_from_csv(padded_resized_csv_path, padded_resized_grayscaled_images_folder, padded_resized_grayscaled_annotations_folder)

Loaded 102 annotations from CSV
Starting annotation process...
Input folder: ./data/teknofest/images/padded-resized-grayscaled
Output folder: ./data/teknofest/annotations/padded-resized-grayscaled
--------------------------------------------------
Progress: 50/102 images annotated
Progress: 100/102 images annotated
--------------------------------------------------
Annotation complete!
Successfully annotated: 102 images
Failed annotations: 0 images
Annotated images saved in: ./data/teknofest/annotations/padded-resized-grayscaled


#### draw on original images (recreate them for visual consistency)

In [ ]:
annotate_images_from_csv(original_csv_path, original_images_folder, original_recreated_annotations_folder)

Loaded 102 annotations from CSV
Starting annotation process...
Input folder: ./data/teknofest/images/original
Output folder: ./data/teknofest/annotations/original-recreated
--------------------------------------------------
Progress: 50/102 images annotated
Progress: 100/102 images annotated
--------------------------------------------------
Annotation complete!
Successfully annotated: 102 images
Failed annotations: 0 images
Annotated images saved in: ./data/teknofest/annotations/original-recreated


## fetch & preprocess jsrt data

data: http://db.jsrt.or.jp/eng.php  
see https://github.com/ngaggion/Chest-xray-landmark-dataset

### save images as png

In [ ]:
def preprocess(folderpath, flist):
    os.makedirs(folderpath, exist_ok=True)
    
    for f in flist:
        try:
            print(f"Processing {f}...")
            with open(f, 'rb') as path:
                dtype = np.dtype('uint16')
                img = np.fromfile(path, dtype=dtype)
                print(f"  Raw size: {img.size}")
                if img.size != 2048 * 2048:
                    print("  Skipping: unexpected image size")
                    continue
                img = img.reshape((2048, 2048))

            img = 1 - img.astype('float32') / 65535.0
            img = cv2.resize(img, (1024, 1024))
            img = (img * 255).astype('uint8')

            p = os.path.join(folderpath, os.path.basename(f).replace('.IMG', '.png'))
            success = cv2.imwrite(p, img)
            if not success:
                print(f"  Failed to save {p}")
        except Exception as e:
            print(f"  Error with {f}: {e}")

def natural_key(string_):
    """See http://www.codinghorror.com/blog/archives/001018.html"""
    return [int(s) if s.isdigit() else s for s in re.split(r'(\d+)', string_)]

In [ ]:
img_path = "All247images"

data_root = pathlib.Path(img_path)
all_files = list(data_root.glob('*.IMG'))
all_files = [str(path) for path in all_files]
all_files.sort(key = natural_key)

save_path = "Images/"
preprocess(save_path, all_files)

### save annotations as png from np arrays

In [ ]:
def draw_organ(ax, array, color = 'b'):
    N = array.shape[0]
    for i in range(0, N):
        x, y = array[i,:]
        circ = plt.Circle((x, y), radius=3, color=color, fill = True)
        ax.add_patch(circ)
    return

def draw_lines(ax, array, color = 'b'):
    N = array.shape[0]
    for i in range(0, N):
        x1, y1 = array[i-1,:]
        x2, y2 = array[i,:]
        ax.plot([x1, x2], [y1, y2], color=color, linestyle='-', linewidth=1)
    return

def drawOrgans(RL, LL, H = None, img =  None):
    fig, ax = plt.subplots()
    
    if img is not None:
        plt.imshow(img, cmap='gray')
    else:
        img = np.zeros([1024, 1024])
        plt.imshow(img)
    
    plt.axis('off')
    
    draw_lines(ax, RL, 'r')
    draw_lines(ax, LL, 'g')
    
    draw_organ(ax, RL, 'r')
    draw_organ(ax, LL, 'g')
    
    if H is not None:
        draw_lines(ax, H, 'y')
        draw_organ(ax, H, 'y')

    return

def getDenseMask(RL, LL, H = None, imagesize = 1024):
    img = np.zeros([1024,1024])
    
    RL = RL.reshape(-1, 1, 2).astype('int')
    LL = LL.reshape(-1, 1, 2).astype('int')

    img = cv2.drawContours(img, [RL], -1, 1, -1)
    img = cv2.drawContours(img, [LL], -1, 2, -1)
    
    if H is not None:
        H = H.reshape(-1, 1, 2).astype('int')
        img = cv2.drawContours(img, [H], -1, 3, -1)
    
    return img

In [ ]:
data_root = pathlib.Path("Images/")
all_files = list(data_root.glob('*.png'))
all_files = [str(path) for path in all_files]
all_files.sort(key = natural_key)

img1 = all_files[0]
RL = img1.replace('Images','landmarks/RL').replace('.png','.npy')
LL = img1.replace('Images','landmarks/LL').replace('.png','.npy')
H = img1.replace('Images','landmarks/H').replace('.png','.npy')

img = cv2.imread(img1,0)
RL = np.load(RL)
LL = np.load(LL)
H = np.load(H)

In [ ]:
output_folder = pathlib.Path("DenseMasks/")
output_folder.mkdir(exist_ok=True)

for img_path in all_files:
    RL_path = img_path.replace('Images', 'landmarks/RL').replace('.png', '.npy')
    LL_path = img_path.replace('Images', 'landmarks/LL').replace('.png', '.npy')
    H_path = img_path.replace('Images', 'landmarks/H').replace('.png', '.npy')

    if not (os.path.exists(RL_path) and os.path.exists(LL_path)):
        print(f"Deleting {img_path} because landmarks are missing.")
        os.remove(img_path)
        continue

    RL = np.load(RL_path)
    LL = np.load(LL_path)
    H = np.load(H_path) if os.path.exists(H_path) else None

    mask = getDenseMask(RL, LL, H)
    mask_img = (mask * 85).astype(np.uint8)  # colors: 0, 85, 170, 255

    filename = pathlib.Path(img_path).stem + "_mask.png"
    output_path = output_folder / filename
    cv2.imwrite(str(output_path), mask_img)

### resize images and annotations to 512x512

In [ ]:
def resize_squared_images_to_square(target_width, input_images_folder_path, output_images_folder_path):
    for image_path in [os.path.join(input_images_folder_path, filename) for filename in os.listdir(input_images_folder_path)]:
        with Image.open(image_path) as img:
            if img.width == img.height:
                resized_img = img.resize((target_width, target_width))
                resized_img.save(os.path.join(output_images_folder_path, os.path.basename(image_path)))

In [ ]:
resize_squared_images_to_square(512, "Images", "images")
resize_squared_images_to_square(512, "DenseMasks", "annotations")

## new